In [47]:
#analysing employee dataset and creating api(s) for the data

In [48]:
import pandas as pd

df = pd.read_csv('Messy_Employee_dataset.csv')
null_values = df.isnull().sum()
print(null_values)

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64


In [49]:
df['Age'] = df['Age'].fillna(0)
df['Salary'] = df['Salary'].fillna(0)
print(df)

     Employee_ID First_Name Last_Name   Age   Department_Region    Status  \
0        EMP1000        Bob     Davis  25.0   DevOps-California    Active   
1        EMP1001        Bob     Brown   0.0       Finance-Texas    Active   
2        EMP1002      Alice     Jones   0.0        Admin-Nevada   Pending   
3        EMP1003        Eva     Davis  25.0        Admin-Nevada  Inactive   
4        EMP1004      Frank  Williams  25.0  Cloud Tech-Florida    Active   
...          ...        ...       ...   ...                 ...       ...   
1015     EMP2015      David    Miller  30.0       HR-California    Active   
1016     EMP2016      David   Johnson  30.0    Cloud Tech-Texas  Inactive   
1017     EMP2017    Charlie  Williams  40.0    Finance-New York    Active   
1018     EMP2018      Alice    Garcia  30.0          HR-Florida  Inactive   
1019     EMP2019      Heidi     Jones  30.0     DevOps-Illinois   Pending   

       Join_Date     Salary                         Email       Phone  \
0 

In [56]:
import sqlite3

conn = sqlite3.connect('employees.db')
df.to_sql('employees', conn, if_exists = 'replace', index = False)
cursor = conn.cursor()
cursor.execute(""" 
SELECT
Salary,
CASE
WHEN Salary < 10000.00 THEN 'Low Income'
WHEN Salary BETWEEN 10000.00 AND 60000.00 THEN 'Medium Income' 
WHEN Salary > 60000.00 THEN 'High Income'
END AS Salary_Category
FROM employees
;
""")
try:
    results = cursor.fetchall()
    data = pd.DataFrame(results)
    print(data)
finally:
    conn.close()

              0              1
0      59767.65  Medium Income
1      65304.66    High Income
2      88145.90    High Income
3      69450.99    High Income
4     109324.61    High Income
...         ...            ...
1015       0.00     Low Income
1016  100215.06    High Income
1017  114587.11    High Income
1018   71318.79    High Income
1019   77764.24    High Income

[1020 rows x 2 columns]


In [52]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

X = df.drop(['Salary', 'Email', 'Phone', 'First_Name', 'Last_Name', 'Employee_ID'], axis = 1)
y = df['Salary']

categorical_features = ['Department_Region', 'Join_Date', 'Status', 'Remote_Work', 'Performance_Score']
numerical_features = ['Age']

preprocessor = ColumnTransformer(transformers = [
    ('cat', StandardScaler(), numerical_features),
    ('num', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.3, random_state = 45)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
predictions_data = pd.DataFrame(predictions)
print(predictions_data)


                 0
0     35790.855040
1     66847.622779
2     82330.568423
3    149877.439043
4     94616.128723
..             ...
301   62492.818414
302   87792.649062
303   82382.252115
304   67757.915779
305   79622.845198

[306 rows x 1 columns]
